Imports & Chargement

In [1]:
import pandas as pd
import numpy as np
import re

from sklearn.preprocessing import OneHotEncoder, LabelEncoder, MinMaxScaler
from sklearn.impute import SimpleImputer

In [2]:
df = pd.read_csv("mainSimulationAccessTraces.csv", low_memory=False)
print(df.shape)
df.head()

(357952, 13)


,sourceID,sourceAddress,sourceType,sourceLocation,destinationServiceAddress,destinationServiceType,destinationLocation,accessedNodeAddress,accessedNodeType,operation,value,timestamp,normality
0,lightcontrol2,/agent2/lightcontrol2,/lightControler,BedroomParents,/agent2/lightcontrol2,/lightControler,BedroomParents,/agent2/lightcontrol2,/lightControler,registerService,none,1520031600000,normal
1,lightcontrol3,/agent3/lightcontrol3,/lightControler,Dinningroom,/agent3/lightcontrol3,/lightControler,Dinningroom,/agent3/lightcontrol3,/lightControler,registerService,none,1520031603269,normal
2,lightcontrol1,/agent1/lightcontrol1,/lightControler,BedroomChildren,/agent1/lightcontrol1,/lightControler,BedroomChildren,/agent1/lightcontrol1,/lightControler,registerService,none,1520031603279,normal
3,lightcontrol4,/agent4/lightcontrol4,/lightControler,Kitchen,/agent4/lightcontrol4,/lightControler,Kitchen,/agent4/lightcontrol4,/lightControler,registerService,none,1520031603290,normal
4,movement4,/agent4/movement4,/movementSensor,Kitchen,/agent4/movement4,/movementSensor,Kitchen,/agent4/movement4,/movementSensor,registerService,none,1520031603464,normal


Target encoding

In [3]:
df['normality'] = df['normality'].apply(
    lambda x: 0 if x == 'normal' else 1
)
df['normality'].value_counts()

normality
0    347935
1     10017
Name: count, dtype: int64

Nettoyage structurel

In [4]:
drop_cols = [
    'sourceID',
    'sourceAddress',
    'destinationServiceAddress',
    'accessedNodeAddress'
]

df.drop(columns=drop_cols, inplace=True)

Typage sémantique de value

In [5]:
JAVA_OBJ_REGEX = re.compile(r"^[a-zA-Z0-9_.]+@[a-f0-9]+$")

def detect_value_type(v):
    if pd.isna(v):
        return "missing"
    v = str(v).strip().lower()
    if v in ("none", "null", "nan", ""):
        return "missing"
    if v in ("true", "false"):
        return "boolean"
    if JAVA_OBJ_REGEX.match(v):
        return "java_object"
    try:
        f = float(v)
        if f > 1e11:
            return "timestamp_ms"
        if v.isdigit():
            return "integer"
        return "float"
    except:
        return "string"

df['value_type'] = df['value'].apply(detect_value_type)
df['value_type'].value_counts()


value_type
integer         184469
float           130115
boolean          40426
missing           2156
timestamp_ms       575
string             200
java_object         11
Name: count, dtype: int64

Features numériques issues de value

In [6]:
df['value_int'] = pd.to_numeric(df['value'], errors='coerce')
df['value_float'] = pd.to_numeric(df['value'], errors='coerce')
df['value_bool'] = df['value'].map({'true':1, 'false':0})

In [7]:
for t in df['value_type'].unique():
    df[f'value_is_{t}'] = (df['value_type'] == t).astype(int)

Features temporelles

In [8]:
df['event_time'] = pd.to_datetime(df['timestamp'], unit='ms', errors='coerce')

df['hour'] = df['event_time'].dt.hour
df['minute'] = df['event_time'].dt.minute
df['second'] = df['event_time'].dt.second

df['time_of_day'] = df['hour'] + df['minute']/60
df['is_night'] = df['hour'].apply(lambda h: 1 if h >= 22 or h <= 6 else 0)

Colonnes catégorielles

In [ ]:
categorical_cols = [
    'sourceType',
    'sourceLocation',
    'destinationServiceType',
    'destinationLocation',
    'accessedNodeType',
    'operation'
]

df[categorical_cols] = df[categorical_cols].fillna('unknown')

Encodage (Tree-based)

In [10]:
encoder = OneHotEncoder(
    sparse_output=False,
    handle_unknown='ignore'
)

encoded = encoder.fit_transform(df[categorical_cols])
encoded_df = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(categorical_cols)
)

df_tree = pd.concat(
    [df.drop(columns=categorical_cols + ['value', 'timestamp', 'event_time', 'value_type']),
     encoded_df],
    axis=1
)

Imputation & normalisation

In [11]:
numeric_cols = [
    'value_int', 'value_float', 'value_bool',
    'hour', 'minute', 'second', 'time_of_day', 'is_night'
]

imputer = SimpleImputer(strategy='median')
df_tree[numeric_cols] = imputer.fit_transform(df_tree[numeric_cols])

scaler = MinMaxScaler()
df_tree[['hour','minute','second','time_of_day']] = scaler.fit_transform(
    df_tree[['hour','minute','second','time_of_day']]
)

Dataset final

In [ ]:
df_tree.to_csv(
    "data/data_preprocessed.csv",
    index=False
)

df_tree.head()

,normality,value_int,value_float,value_bool,value_is_missing,value_is_float,value_is_integer,value_is_timestamp_ms,value_is_boolean,value_is_java_object,...,accessedNodeType_/sensorService,accessedNodeType_/smartPhone,accessedNodeType_/thermostat,accessedNodeType_/washingService,accessedNodeType_unknown,operation_lockSubtree,operation_read,operation_registerService,operation_subscribe,operation_write
0,0,7.0,7.0,0.0,1,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,0,7.0,7.0,0.0,1,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,0,7.0,7.0,0.0,1,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,0,7.0,7.0,0.0,1,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,0,7.0,7.0,0.0,1,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
